# Accel event sim

Thin front-end over the `accel` package (physics lives there; see `accel/sim.py` docstring).
Compares three runs: **ideal TC** (quasi-static, force = min(motor, peak grip) — the old joe-accel model),
**no TC**, and **TC** (slip-target PI in `accel/control.py`).

Legacy version: `accel_weight_transfer.ipynb`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from accel import Car, CurveTire, MotorMap, PacejkaTire, SlipTC, quasi_static, simulate

plt.style.use("bmh")
motor = MotorMap("228HV")
tire = CurveTire(mu_peak=1.4)          # or PacejkaTire() for the TTC LC0 fit
car = Car(final_drive=3.5)
tc = SlipTC.for_tire(tire)

runs = [quasi_static(car, motor, tire=tire), simulate(car, motor, tire=tire),
        simulate(car, motor, tire=tire, tc=tc)]
pd.DataFrame([r.summary() for r in runs])

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 15), sharex=True)
for r in runs:
    d = r.df
    axes[0].plot(d.t, d.v * 3.6, label=r.label)
    axes[1].plot(d.t, d.a / 9.81, label=r.label)
    axes[2].plot(d.t, d.motor_torque, label=r.label)
    axes[4].plot(d.t, d.bus_v, label=r.label)
for r in runs[1:]:
    axes[3].plot(r.df.t, r.df.slip.clip(upper=1.0), label=r.label)
axes[3].axhline(tc.target_slip, color="r", ls="--", label="TC target")
for ax, y in zip(axes, ["speed (km/h)", "accel (g)", "motor torque (Nm)", "slip ratio", "bus voltage (V)"]):
    ax.set_ylabel(y); ax.legend()
axes[-1].set_xlabel("time (s)")
plt.tight_layout()

## Launch detail: what TC is doing

In [ ]:
d = runs[2].df.query("t < 1.0")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(d.t, d.slip, label="measured slip (TC input)")
ax1.plot(d.t, d.kappa, label="tire slip (transient)")
ax1.axhline(tc.target_slip, color="r", ls="--")
ax1.set_ylim(-0.1, 0.6); ax1.legend()
ax2.plot(d.t, d.tc_scale, label="TC torque scale")
ax2.plot(d.t, d.fz_rear / d.fz_rear.max(), label="rear Fz (normalized)")
ax2.legend(); ax2.set_xlabel("time (s)")
plt.tight_layout()

## Final drive sweep

In [ ]:
rows = []
for fd in [2.4, 2.8, 3.2, 3.6, 4.0, 4.5, 5.0]:
    c = Car(final_drive=fd)
    for r in [quasi_static(c, motor, tire=tire), simulate(c, motor, tire=tire),
              simulate(c, motor, tire=tire, tc=tc)]:
        rows.append({"fd": fd, "label": r.label, "time": r.accel_time})
sweep = pd.DataFrame(rows).pivot(index="fd", columns="label", values="time")
ax = sweep.plot(marker="o", figsize=(10, 5), ylabel="accel time (s)", title=f"{motor.name}, {type(tire).__name__}")
sweep

## Validation: KS5e launches

Real launches from `cardata/ks5e` (logger segments, same filter as the old notebook), lined up at torque
application. The logs have no wheel speed or IMU: motor rpm **is** rear wheel speed, so it shows wheelspin
directly. The sim is set up as KS5e (`KS5E_SETUP`: 228HV, 60s8p HE2 pack, FD 3.0 — motor/pack inferred
from the logs, FD assumed). Dashed red is the TC sim's ground speed in motor rpm: the gap to its wheel
line is the slip.

In [ ]:
import math
from accel.compare import KS5E_SETUP, OPTIMUMLAP_SETUP, load_ks5e_runs, load_optimumlap, time_to

ks5e_runs = load_ks5e_runs()
ks5e_car = KS5E_SETUP["car"]
to_rpm = ks5e_car.final_drive / (2 * math.pi * ks5e_car.tire_radius) * 60
lc0 = PacejkaTire()
ks5e_sims = [
    (simulate(**KS5E_SETUP, label="no TC, curve tire"), "k"),
    (simulate(**KS5E_SETUP, tire=lc0, label="no TC, LC0 tire"), "b"),
    (simulate(**KS5E_SETUP, tc=SlipTC(), label="TC"), "r"),
]

T = 1.5
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for r in ks5e_runs:
    d = r.query("t <= @T")
    ax1.plot(d.t, d.motor_rpm, lw=1, alpha=0.7, label=r.attrs["label"])
    ax2.plot(d.t, d.motor_torque, lw=1, alpha=0.7)
for res, c in ks5e_sims:
    d = res.df.query("t <= @T")
    ax1.plot(d.t, d.motor_rpm, c, lw=2.5, label=f"sim {res.label}")
    ax2.plot(d.t, d.motor_torque, c, lw=2.5, label=f"sim {res.label}")
d = ks5e_sims[-1][0].df.query("t <= @T")
ax1.plot(d.t, d.v * to_rpm, "r--", lw=2, label="sim TC: ground speed")
ax1.set_ylabel("motor rpm (= rear wheel speed)"); ax2.set_ylabel("motor torque (Nm)")
ax2.set_xlabel("time from torque application (s)")
ax1.legend(fontsize=7, ncol=2); ax2.legend(fontsize=8)
ax1.set_title("Launch: KS5e logs vs sim")
plt.tight_layout()

Whole runs (short pulls end where the driver lifted). Real v/x assume no wheelspin, so times are only meaningful for runs that hooked up.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
cols = ["motor_rpm", "motor_torque", "bus_v", "bus_i"]
for r in ks5e_runs:
    for ax, c in zip(axes, cols):
        ax.plot(r.t, r[c], ".", ms=2, alpha=0.6, label=r.attrs["label"])
for res, c in ks5e_sims:
    for ax, col in zip(axes, cols):
        ax.plot(res.df.t, res.df[col], c, lw=2, label=f"sim {res.label}")
for ax, y in zip(axes, ["motor rpm", "motor torque (Nm)", "bus voltage (V)", "bus current (A)"]):
    ax.set_ylabel(y)
axes[0].legend(fontsize=7, ncol=3); axes[-1].set_xlabel("time from torque application (s)")
plt.tight_layout()

pd.DataFrame([{"run": r.attrs["label"], "duration_s": round(r.t.max(), 2),
               "timed_run_s (0.3 -> 75.3 m)": round(time_to(r, 75.3, 0.3), 3)} for r in ks5e_runs]
             + [{"run": f"sim {res.label}", "duration_s": None,
                 "timed_run_s (0.3 -> 75.3 m)": round(res.accel_time, 3)} for res, _ in ks5e_sims])

## Validation: OptimumLap

OptimumLap's 208MV test strip vs the sim with the same setup (`OPTIMUMLAP_SETUP`: 208MV, ratio 4.0).
OptimumLap is a point-mass model with its own grip settings, so this is a cross-check, not ground truth.
It also starts at 3.6 km/h and times from x = 0, so times below are 0 -> 75 m for both.

In [ ]:
olap = load_optimumlap()
olap_sims = [quasi_static(**OPTIMUMLAP_SETUP), simulate(**OPTIMUMLAP_SETUP, tc=SlipTC())]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
axes[0].plot(olap.x, olap.v * 3.6, "k", lw=2, label="OptimumLap")
axes[1].plot(olap.x, olap.a / 9.81, "k", lw=2, label="OptimumLap")
axes[2].plot(olap.x, olap.motor_torque, "k", lw=2, label="OptimumLap (net of its TC)")
for res in olap_sims:
    d = res.df
    axes[0].plot(d.x, d.v * 3.6, label=f"sim {res.label}")
    axes[1].plot(d.x, d.a / 9.81, label=f"sim {res.label}")
    axes[2].plot(d.x, d.motor_torque, label=f"sim {res.label}")
for ax, y in zip(axes, ["speed (km/h)", "accel (g)", "motor torque (Nm)"]):
    ax.set_ylabel(y); ax.legend()
axes[-1].set_xlabel("distance (m)"); axes[-1].set_xlim(0, 75)
plt.tight_layout()

pd.DataFrame([{"run": "OptimumLap", "t_0_to_75m": round(time_to(olap, 75), 3)}]
             + [{"run": f"sim {r.label}", "t_0_to_75m": round(time_to(r.df, 75), 3)} for r in olap_sims])